# CIVID Master Dashboard
### Civilian Impact Verified Incident Dataset — Combined Phase 1 (Palestine/Gaza) & Phase 2 (Sudan)

**Scope:** This notebook loads all verified events, person records, and sources from both
completed phases and produces summary statistics and visualizations.

**Data limitations:**
- Some entries carry `verification_status = estimated` rather than `verified` — treat these
  with appropriate caution.
- Phase 2 second batch is source-concentrated (ICRC/ACLED heavy) — see `sources.csv` batch note.
- No images are displayed unless the cited source itself provided a licensed image of
  infrastructure/aid response. No victim or casualty imagery is included, per project ethics rules.

Run all cells top to bottom. Requires: `pandas`, `matplotlib`, `seaborn` (see `environment.yml`).

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.dpi"] = 110

In [20]:
from pathlib import Path
import pandas as pd

# 1. Locate repository root dynamically
cwd = Path.cwd()
root = cwd if (cwd / "data").exists() else cwd.parent

# 2. Define data paths mapped to their phase identifiers
phase_paths = {
    "Palestine/Gaza": root / "data" / "phase1_palestine",
    "Sudan": root / "data" / "phase2_sudan",
}

# 3. Initialize mapping buckets for the DataFrames
data_buckets = {
    "events.csv": [],
    "persons.csv": [],
    "sources.csv": []
}

# 4. Ingest and tag CSV files (with error handling for malformed rows)
for phase_name, phase_dir in phase_paths.items():
    for fname, bucket in data_buckets.items():
        fpath = phase_dir / fname
        if fpath.exists():
            try:
                # 'on_bad_lines="warn"' skips corrupted lines and prints a warning
                df = pd.read_csv(fpath, on_bad_lines="warn")
                df["phase_name"] = phase_name
                bucket.append(df)
            except Exception as e:
                print(f"[error] Failed to parse {fpath}: {e}")
        else:
            print(f"[warn] Missing expected file: {fpath}")

# 5. Concatenate DataFrames safely
events = pd.concat(data_buckets["events.csv"], ignore_index=True) if data_buckets["events.csv"] else pd.DataFrame()
persons = pd.concat(data_buckets["persons.csv"], ignore_index=True) if data_buckets["persons.csv"] else pd.DataFrame()
sources = pd.concat(data_buckets["sources.csv"], ignore_index=True) if data_buckets["sources.csv"] else pd.DataFrame()

# 6. Sanitize numeric columns in 'events'
numeric_cols = ["fatalities", "injuries", "missing"]
for col in numeric_cols:
    if col in events.columns:
        events[col] = pd.to_numeric(events[col], errors="coerce")

# 7. Summary Report
print(f"Loaded: {len(events)} events, {len(persons)} person records, {len(sources)} source rows")

Loaded: 38 events, 9 person records, 19 source rows


C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_16496\54984210.py:28: ParserWarning: Skipping line 10: expected 15 fields, saw 16

  df = pd.read_csv(fpath, on_bad_lines="warn")


## 1. Dataset Overview

In [21]:
summary = pd.DataFrame({
    "metric": [
        "Total events", "Verified events", "Estimated events",
        "Total fatalities", "Total injuries", "Total missing",
        "Person records", "Distinct sources"
    ],
    "value": [
        len(events),
        int((events["verification_status"] == "verified").sum()) if "verification_status" in events else "n/a",
        int((events["verification_status"] == "estimated").sum()) if "verification_status" in events else "n/a",
        int(events["fatalities"].fillna(0).sum()) if "fatalities" in events else "n/a",
        int(events["injuries"].fillna(0).sum()) if "injuries" in events else "n/a",
        int(events["missing"].fillna(0).sum()) if "missing" in events else "n/a",
        len(persons),
        sources["source_name"].nunique() if "source_name" in sources else "n/a",
    ],
})
display(summary)

,metric,value
0,Total events,38
1,Verified events,20
2,Estimated events,3
3,Total fatalities,31
4,Total injuries,132
5,Total missing,0
6,Person records,9
7,Distinct sources,16


## 2. Events by Country (Phase Comparison)

In [22]:
if "phase_name" in events.columns and len(events):
    by_country = events["phase_name"].value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    by_country.plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"])
    ax.set_title("Verified Events by Country/Phase")
    ax.set_ylabel("Number of events")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No data available.")

C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_16496\3087767769.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Events by Category

In [23]:
cat_col = "conflict_name" if "conflict_name" in events.columns else None
# Fallback: try a category-like column if present
for candidate in ["category", "event_category", "location_type"]:
    if candidate in events.columns:
        cat_col = candidate
        break

if cat_col and len(events):
    fig, ax = plt.subplots(figsize=(8, 5))
    events[cat_col].value_counts().plot(kind="barh", ax=ax, color="#55A868")
    ax.set_title(f"Events by {cat_col}")
    ax.set_xlabel("Number of events")
    plt.tight_layout()
    plt.show()
else:
    print("No category-like column found — adjust cat_col above to match your schema.")

C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_16496\2267870095.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Timeline (Chronological, Both Phases)

In [24]:
if "event_date" in events.columns and len(events):
    events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")
    timeline = events.dropna(subset=["event_date"]).sort_values("event_date")
    fig, ax = plt.subplots(figsize=(10, 4))
    for phase, color in zip(timeline["phase_name"].unique(), ["#4C72B0", "#C44E52"]):
        sub = timeline[timeline["phase_name"] == phase]
        ax.scatter(sub["event_date"], [phase] * len(sub), label=phase, s=60, color=color)
    ax.set_title("Event Timeline")
    ax.set_xlabel("Date")
    plt.xticks(rotation=30, ha="right")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No event_date column found.")

C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_16496\3360273361.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Verification Status Breakdown

In [25]:
if "verification_status" in events.columns and len(events):
    vc = events["verification_status"].value_counts()
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(vc.values, labels=vc.index, autopct="%1.0f%%", colors=["#55A868", "#DD8452", "#C44E52", "#8172B2"])
    ax.set_title("Verification Status")
    plt.tight_layout()
    plt.show()
else:
    print("No verification_status column found.")

C:\Users\Muhammad Anas\AppData\Local\Temp\ipykernel_16496\3265460465.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Source Reliability Table

In [26]:
cols_present = [c for c in ["source_name", "source_type", "reliability_score", "phase_name"] if c in sources.columns]
if cols_present:
    display(sources[cols_present].drop_duplicates().sort_values(cols_present[0]))
else:
    display(sources.head())

,source_name,source_type,reliability_score,phase_name
4,ACLED Conflict Data,conflict event database,0.90,Palestine/Gaza
17,ACLED Sudan Conflict Events Database,conflict event database,0.89,Sudan
8,Country-Based Pooled Funds (OCHA),HDX,NaN,Palestine/Gaza
9,Food and Agriculture Organization (FAO) of the United Nations,HDX,NaN,Palestine/Gaza
13,ICRC Sudan Operations and Humanitarian Response,humanitarian agency,0.94,Sudan
10,Insecurity Insight,HDX,NaN,Palestine/Gaza
5,OCHA occupied Palestinian territory (oPt),HDX,NaN,Palestine/Gaza
3,OHCHR Occupied Palestinian Territory,rights monitoring,0.92,Palestine/Gaza
7,Palestinian Central Bureau of Statistics,HDX,NaN,Palestine/Gaza
18,Phase 2 batch note - ICRC/ACLED heavy extraction,internal_note,0.50,Sudan


## 7. Licensed Media (Infrastructure/Aid Response Only)

Per project ethics rules, this section only displays images that:
- Come directly from the cited source report (ICRC, OCHA, UNRWA, ACLED, etc.)
- Depict infrastructure, aid distribution, or response activity — **never** identifiable
  victims or casualties
- Include a documented license/source in `image_license` and `image_source`

If no such images exist in the current data, this section will show a notice instead
of attempting to fetch or substitute any image.

In [27]:
if "image_available" in persons.columns:
    image_candidates = persons[persons["image_available"] == True]
else:
    image_candidates = pd.DataFrame()

if image_candidates.empty:
    display(Markdown("**No licensed images are present in the current dataset.** "
                      "Image fields remain blank per the project's media-safety policy."))
else:
    show_cols = [c for c in ["phase_name", "event_id", "image_url", "image_source", "image_license", "image_caption"] if c in image_candidates.columns]
    display(image_candidates[show_cols])

**No licensed images are present in the current dataset.** Image fields remain blank per the project's media-safety policy.

---
### Notes
- This dashboard reads directly from the CSVs in `data/`. Re-run all cells after any data update.
- Data quality caveats and source concentration issues are logged in `sources.csv` batch notes.
- See `data_dictionary.md` for full column definitions and `README.md` for project scope and methodology.